# Checkpoint 45 — Retention Cost Model

This notebook reviews the transparent financial and operational assumptions created by `src/calculate_retention_economics.py`. It does not load attrition outcomes, model probabilities, or the reserved test target.

In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"
assumptions = pd.read_csv(processed_dir / "retention_cost_assumption_table.csv")
scenarios = pd.read_csv(processed_dir / "retention_cost_scenarios.csv")
population_summary = pd.read_csv(processed_dir / "retention_cost_population_summary.csv")
validation = pd.read_csv(processed_dir / "retention_cost_validation.csv")

## 1. Assumptions

The three assumption families are kept separate so that financial and operational uncertainty remains visible.

In [2]:
assumptions

,assumption_family,scenario,salary_multiplier,intervention_cost_usd,success_probability,vacancy_days,time_to_productivity_days,training_hours_per_replacement,coverage_hours_per_replacement
0,Replacement impact,Low,0.5,NaN,NaN,30.0,60.0,40.0,60.0
1,Replacement impact,Base,1.0,NaN,NaN,60.0,90.0,80.0,180.0
2,Replacement impact,High,1.5,NaN,NaN,90.0,150.0,120.0,360.0
3,Intervention cost,Lean,NaN,1000.0,NaN,NaN,NaN,NaN,NaN
4,Intervention cost,Standard,NaN,2500.0,NaN,NaN,NaN,NaN,NaN
5,Intervention cost,Intensive,NaN,5000.0,NaN,NaN,NaN,NaN,NaN
6,Intervention effectiveness,Conservative,NaN,NaN,0.10,NaN,NaN,NaN,NaN
7,Intervention effectiveness,Base,NaN,NaN,0.25,NaN,NaN,NaN,NaN
8,Intervention effectiveness,Optimistic,NaN,NaN,0.40,NaN,NaN,NaN,NaN


## 2. Reference scenario

The reference scenario uses a 100% salary replacement cost, a $2,500 intervention, and 25% intervention effectiveness. It is a comparison anchor, not a final policy.

In [3]:
reference = scenarios.loc[
    scenarios["replacement_impact_scenario"].eq("Base")
    & scenarios["intervention_cost_scenario"].eq("Standard")
    & scenarios["effectiveness_scenario"].eq("Base")
]

reference[[
    "population",
    "population_rows",
    "median_replacement_cost_usd",
    "intervention_cost_per_employee_usd",
    "intervention_success_probability",
    "median_break_even_attrition_probability",
]]

,population,population_rows,median_replacement_cost_usd,intervention_cost_per_employee_usd,intervention_success_probability,median_break_even_attrition_probability
6,Current active,7305,90900.0,2500.0,0.25,0.110011
33,Validation,5521,86400.0,2500.0,0.25,0.115741


## 3. Department cost profile

These are salary-dependent replacement-cost summaries. They do not contain risk scores and should not be interpreted as department performance rankings.

In [4]:
population_summary.sort_values(
    ["population", "average_reference_replacement_cost_usd"],
    ascending=[True, False],
)

,population,snapshot_date,department_name,employees,average_base_salary,median_base_salary,reference_replacement_cost_multiplier,average_reference_replacement_cost_usd,median_reference_replacement_cost_usd
1,Current active,2026-06-30,Engineering,1512,125237.896825,119300.0,1.0,125237.896825,119300.0
4,Current active,2026-06-30,Information Technology,734,115305.585831,113950.0,1.0,115305.585831,113950.0
7,Current active,2026-06-30,Supply Chain,904,88671.238938,83800.0,1.0,88671.238938,83800.0
5,Current active,2026-06-30,Manufacturing,1808,88464.159292,101750.0,1.0,88464.159292,101750.0
2,Current active,2026-06-30,Finance,570,88222.982456,85600.0,1.0,88222.982456,85600.0
3,Current active,2026-06-30,Human Resources,503,84654.274354,83600.0,1.0,84654.274354,83600.0
6,Current active,2026-06-30,Sales,739,83224.763194,78200.0,1.0,83224.763194,78200.0
0,Current active,2026-06-30,Customer Support,535,57535.887850,56400.0,1.0,57535.887850,56400.0
9,Validation,2024-06-30,Engineering,1153,122616.652212,115300.0,1.0,122616.652212,115300.0
12,Validation,2024-06-30,Information Technology,536,111099.813433,113050.0,1.0,111099.813433,113050.0


## 4. Validation

Every row should be `PASS`. In particular, the final rows confirm that no model probabilities, operating threshold, employee ranking, or reserved test outcomes were used.

In [5]:
validation

,check,status,observed,requirement,details
0,Replacement scenarios are complete and unique,PASS,"['Low', 'Base', 'High']",3 unique scenarios,"Low, base, and high replacement impacts must a..."
1,Intervention cost scenarios are complete and u...,PASS,"['Lean', 'Standard', 'Intensive']",3 unique scenarios,The model must compare more than one intervent...
2,Effectiveness scenarios are complete and unique,PASS,"['Conservative', 'Base', 'Optimistic']",3 unique scenarios,Uncertain intervention success must be handled...
3,Replacement-cost range spans 50% to 150% of sa...,PASS,0.50–1.50,0.50–1.50,The committed sensitivity range matches the re...
4,Intervention costs are positive and ordered,PASS,"[1000.0, 2500.0, 5000.0]",Positive increasing costs,Every intervention has an explicit per-employe...
5,Effectiveness assumptions are valid probabilities,PASS,"[0.1, 0.25, 0.4]","Increasing values inside (0, 1]",Effectiveness is uncertain and cannot exceed c...
6,Operational assumptions increase by impact sce...,PASS,"['Low', 'Base', 'High']",Nondecreasing Low to Base to High,More severe scenarios should not assume smalle...
7,Economic populations are sufficiently large,PASS,"{'Current active': 7305, 'Validation': 5521}",Each population >= 1000,Cost summaries need useful validation and curr...
8,Only configured economic snapshots are used,PASS,"{'Current active': ['2026-06-30'], 'Validation...","{'Validation': '2024-06-30', 'Current active':...",Validation economics precede current operation...
9,Population employee-snapshot keys are unique,PASS,0,0 duplicates,One employee contributes once to each economic...
